<h4>This notebook will load a dataset that has finance news reports and we will convert them into chunks -> embed the chunks-> load into a vector database</h4>

In [331]:
# !pip install pinecone
# !pip install openai
# !pip install langchain
# !pip install kagglehub
# !pip install -U langchain-community
!pip install rapidfuzz

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   -------------------------------- ------- 1.3/1.6 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 6.2 MB/s eta 0:00:00


In [2]:
import kagglehub
import os
import pinecone
import openai
import pandas as pd
import numpy as np
import langchain
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Pinecone
from rapidfuzz import process

In [ ]:


# Download latest version
path = kagglehub.dataset_download("notlucasp/financial-news-headlines")

print("Path to dataset files:", path)

100%|██████████| 3.91M/3.91M [00:01<00:00, 2.45MB/s]


Extracting files...
Path to dataset files: C:\Users\cyber\.cache\kagglehub\datasets\notlucasp\financial-news-headlines\versions\2


In [4]:
path = "C:\\Users\\cyber\\.cache\\kagglehub\\datasets\\notlucasp\\financial-news-headlines\\versions\\2"

In [5]:
df_cnbc = pd.read_csv(path + "/cnbc_headlines.csv")
df_gaurdian = pd.read_csv(path + "/guardian_headlines.csv")
df_reuters = pd.read_csv(path + "/reuters_headlines.csv")


In [31]:
df_symbols_valid = pd.read_csv("C:/Users/cyber/.cache/kagglehub/datasets/jacksoncrow/stock-market-dataset/versions/2/symbols_valid_meta.csv")

In [6]:
print(df_cnbc.describe())
print(df_gaurdian.describe())
print(df_reuters.describe())

                                                Headlines  \
count                                                2800   
unique                                               2788   
top     Cramer: I helped investors through the 2010 fl...   
freq                                                    2   

                                 Time  \
count                            2800   
unique                           2474   
top      8:11  PM ET Fri,  2 Nov 2018   
freq                                6   

                                              Description  
count                                                2800  
unique                                               2618  
top     "Mad Money" host Jim Cramer rings the lightnin...  
freq                                                  147  
             Time                                          Headlines
count       17800                                              17800
unique        774                                 

<h4>So the number of entries from all the sources are decent enough to be stored in a database</h4>

In [46]:
df_gaurdian.head()


,Time,Headlines
0,18-Jul-20,Johnson is asking Santa for a Christmas recovery
1,18-Jul-20,‘I now fear the worst’: four grim tales of wor...
2,18-Jul-20,Five key areas Sunak must tackle to serve up e...
3,18-Jul-20,Covid-19 leaves firms ‘fatally ill-prepared’ f...
4,18-Jul-20,The Week in Patriarchy \n\n\n Bacardi's 'lad...


In [45]:
df_cnbc.head()

,Headlines,Time,Description
0,Jim Cramer: A better way to invest in the Covi...,"7:51 PM ET Fri, 17 July 2020","""Mad Money"" host Jim Cramer recommended buying..."
1,Cramer's lightning round: I would own Teradyne,"7:33 PM ET Fri, 17 July 2020","""Mad Money"" host Jim Cramer rings the lightnin..."
2,NaN,NaN,NaN
3,"Cramer's week ahead: Big week for earnings, ev...","7:25 PM ET Fri, 17 July 2020","""We'll pay more for the earnings of the non-Co..."
4,IQ Capital CEO Keith Bliss says tech and healt...,"4:24 PM ET Fri, 17 July 2020","Keith Bliss, IQ Capital CEO, joins ""Closing Be..."


In [47]:
df_reuters.head(5)

,Headlines,Time,Description
0,TikTok considers London and other locations fo...,Jul 18 2020,TikTok has been in discussions with the UK gov...
1,Disney cuts ad spending on Facebook amid growi...,Jul 18 2020,Walt Disney has become the latest company to ...
2,Trail of missing Wirecard executive leads to B...,Jul 18 2020,Former Wirecard chief operating officer Jan M...
3,Twitter says attackers downloaded data from up...,Jul 18 2020,Twitter Inc said on Saturday that hackers were...
4,U.S. Republicans seek liability protections as...,Jul 17 2020,A battle in the U.S. Congress over a new coron...


In [50]:
print("cnbc", df_cnbc['Time'].head(), sep="\n", end="\n\n")
print("guardian", df_gaurdian['Time'].head(), sep="\n", end="\n\n")
print("reuters", df_reuters['Time'].head(), sep="\n", end="\n\n")


cnbc
0     7:51  PM ET Fri, 17 July 2020
1     7:33  PM ET Fri, 17 July 2020
2                               NaN
3     7:25  PM ET Fri, 17 July 2020
4     4:24  PM ET Fri, 17 July 2020
Name: Time, dtype: object

guardian
0    18-Jul-20
1    18-Jul-20
2    18-Jul-20
3    18-Jul-20
4    18-Jul-20
Name: Time, dtype: object

reuters
0    Jul 18 2020
1    Jul 18 2020
2    Jul 18 2020
3    Jul 18 2020
4    Jul 17 2020
Name: Time, dtype: object



<h4>So the date in the cnbc needs to be formatted to just the date as the time doesn't matter, guardian is in a different format so that's needs to be changed as well only reuters is in the correct format

In [7]:
df_reuters = df_reuters.rename(columns={"Time": "Date"})  
#convert to datetime
df_reuters['Date'] = pd.to_datetime(df_reuters['Date'])
df_reuters['Date'] = df_reuters['Date'].dt.strftime('%Y-%m-%d')
df_reuters['Date'].head(5) #check if the conversion was successful

0    2020-07-18
1    2020-07-18
2    2020-07-18
3    2020-07-18
4    2020-07-17
Name: Date, dtype: object

In [8]:
df_reuters.isnull().sum() #check for null values

Headlines      0
Date           0
Description    0
dtype: int64

So only the first 3 letters of every month, convert every month into numbers then convert to dateTime

In [9]:
df_gaurdian.dropna(how='any', inplace=True) #drop null values

In [13]:
df_gaurdian = df_gaurdian.rename(columns={"Time": "Date"})  

In [14]:
df_gaurdian[df_gaurdian['Date'] == '18']

,Date,Headlines


In [15]:
months = [
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
]
new = {
    'fine' : [],
    'wrong' : []
}
df_gaurdian_date_fixed = df_gaurdian['Date'].str.split('-').apply(lambda x: f"1-{x[0]}-{x[1]}" if  x[1] not in months else f"{x[0]}-{x[1]}-{x[2]}")
print(new['wrong'], len(new['wrong']), sep="\n")

[]
0


In [16]:
df_gaurdian_date_fixed.head()

0    18-Jul-20
1    18-Jul-20
2    18-Jul-20
3    18-Jul-20
4    18-Jul-20
Name: Date, dtype: object

In [17]:
months = {
    'Jan': '01',
    'Feb': '02',
    'Mar': '03',
    'Apr': '04',
    'May': '05',
    'Jun': '06',
    'Jul': '07',
    'Aug': '08',
    'Sep': '09',
    'Oct': '10',
    'Nov': '11',
    'Dec': '12'
}

df_gaurdian_temp = df_gaurdian_date_fixed.str.split('-').apply(lambda x:f"20{x[2]}-{months[x[1]]}-{x[0]}" )

df_gaurdian_temp.head(5) #check if the conversion was successful

0    2020-07-18
1    2020-07-18
2    2020-07-18
3    2020-07-18
4    2020-07-18
Name: Date, dtype: object

In [18]:
df_gaurdian['Date'] = df_gaurdian_temp

In [19]:
df_gaurdian.head()

,Date,Headlines
0,2020-07-18,Johnson is asking Santa for a Christmas recovery
1,2020-07-18,‘I now fear the worst’: four grim tales of wor...
2,2020-07-18,Five key areas Sunak must tackle to serve up e...
3,2020-07-18,Covid-19 leaves firms ‘fatally ill-prepared’ f...
4,2020-07-18,The Week in Patriarchy \n\n\n Bacardi's 'lad...


In [20]:
df_cnbc['Time'].describe()
df_cnbc['Time'].isnull().sum()
#Get rows where time is null and headlines is not null
df_cnbc[df_cnbc['Time'].isnull() & df_cnbc['Headlines'].notnull()]
#drop these rows
df_cnbc.dropna(subset=['Time'], inplace=True) #drop null values
df_cnbc['Time'].isnull().sum() #check if the conversion was successful

0

In [21]:
df_cnbc_temp = df_cnbc['Time'].str.split(',').apply(lambda x: x[1].lstrip() )
# df_cnbc_temp = df_cnbc_temp.str.split(',').apply(lambda x:f"20{x[2]}-{months[x[1]]}-{x[0]}")
df_cnbc_temp.head(5)

0    17 July 2020
1    17 July 2020
3    17 July 2020
4    17 July 2020
5    16 July 2020
Name: Time, dtype: object

In [22]:
months = [
    'Jan', 'Feb', 'March', 'April', 'May', 'June',
    'July', 'Aug', 'Sept', 'Oct', 'Nov', 'Dec'
]
new = {
    'fine' : [],
    'wrong' : []
}
df_cnbc_date_fixed = df_cnbc_temp.str.split(' ').apply(lambda x: f"{x[2]}-{x[1]}-{x[0]}")
df_cnbc_date_fixed.head(5) #check if the conversion was successful

0    2020-July-17
1    2020-July-17
3    2020-July-17
4    2020-July-17
5    2020-July-16
Name: Time, dtype: object

In [23]:
print(new['fine'][:5], len(new['fine']), sep="\n", end="\n\n")

[]
0



In [24]:
months = {
    'Jan': '01',
    'Feb': '02',
    'March': '03',
    'April': '04',
    'May': '05',
    'June': '06',
    'July': '07',
    'Aug': '08',
    'Sept': '09',
    'Oct': '10',
    'Nov': '11',
    'Dec': '12'
}

df_cnbc_temp = df_cnbc_date_fixed.str.split('-').apply(lambda x:f"{x[0]}-{months[x[1]]}-{x[2]}" )

df_cnbc_temp.head(5) #check if the conversion was successful

0    2020-07-17
1    2020-07-17
3    2020-07-17
4    2020-07-17
5    2020-07-16
Name: Time, dtype: object

In [25]:
df_cnbc['Date'] = df_cnbc_temp
df_cnbc.drop(columns=['Time'], inplace=True) #drop the old time column


In [26]:
#convert to datetime
df_cnbc['Date'] = pd.to_datetime(df_cnbc['Date'])
df_cnbc['Date'] = df_cnbc['Date'].dt.strftime('%Y-%m-%d')
df_cnbc['Date'].head(5) #check if the conversion was successful

0    2020-07-17
1    2020-07-17
3    2020-07-17
4    2020-07-17
5    2020-07-16
Name: Date, dtype: object

In [27]:
df_gaurdian['Date'] = pd.to_datetime(df_gaurdian['Date'])
df_gaurdian['Date'] = df_gaurdian['Date'].dt.strftime('%Y-%m-%d')

In [28]:
df_cnbc.sort_values(by='Date', inplace=True)
df_gaurdian.sort_values(by='Date', inplace=True)    
df_reuters.sort_values(by='Date', inplace=True)

In [29]:
df_reuters['Date'].head(5)

32769    2018-03-20
32738    2018-03-20
32737    2018-03-20
32736    2018-03-20
32735    2018-03-20
Name: Date, dtype: object

<h4> Now we need to add a "Ticker" Coloumn so that it becomes easy for the vector db to fetch the data later</h4>

In [32]:
df_symbols_valid.head(5)



,Nasdaq Traded,Symbol,Security Name,Listing Exchange,Market Category,ETF,Round Lot Size,Test Issue,Financial Status,CQS Symbol,NASDAQ Symbol,NextShares
0,Y,A,"Agilent Technologies, Inc. Common Stock",N,,N,100.0,N,NaN,A,A,N
1,Y,AA,Alcoa Corporation Common Stock,N,,N,100.0,N,NaN,AA,AA,N
2,Y,AAAU,Perth Mint Physical Gold ETF,P,,Y,100.0,N,NaN,AAAU,AAAU,N
3,Y,AACG,ATA Creativity Global - American Depositary Sh...,Q,G,N,100.0,N,N,NaN,AACG,N
4,Y,AADR,AdvisorShares Dorsey Wright ADR ETF,P,,Y,100.0,N,NaN,AADR,AADR,N


In [33]:
df_symbols_valid_temp = df_symbols_valid[df_symbols_valid['ETF'] == 'N'] #filter out ETFs

In [34]:
df_symbols_valid_temp = df_symbols_valid_temp[['Symbol', 'Security Name']].copy() #keep only the relevant columns

In [39]:
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('Common Stock', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('Depositary Shares', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('Units', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('Limited', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('Ordinary Shares', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('technologies', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('corporation', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('inc', '') #remove common stock from the security name
df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.replace('group', '') #remove common stock from the security name

df_symbols_valid_temp['Security Name'] = df_symbols_valid_temp['Security Name'].str.lower()

In [40]:
df_symbols_valid_temp

,Symbol,Security Name
0,A,"agilent , ."
1,AA,alcoa
3,AACG,"ata creativity global - american , each repres..."
5,AAL,"american airlines , . -"
6,AAMC,altisource asset management corp com
...,...,...
8044,ZUO,"zuora, . class a"
8045,ZVO,zovio . -
8046,ZYME,zymeworks . common shares
8047,ZYNE,"zynerba pharmaceuticals, . -"


In [44]:
i =0 
symbolAndSecurityName = {
}
for j, symbol_row in df_symbols_valid_temp.iterrows():
    symbolAndSecurityName[f"{symbol_row['Symbol']}"] = []
    symbolAndSecurityName[f"{symbol_row['Symbol']}"].append(symbol_row['Security Name'])

symbolAndSecurityName['MSFT']

['microsoft  - ']

In [47]:
for i in symbolAndSecurityName.values():
    print(i)
    i[0] = i[0].replace(' ', '')

['agilent , . ']
['alcoa   ']
['ata creativity global - american , each representing two common shares']
['american airlines , . - ']
['altisource asset management corp com']
['atlantic american  - ']
["aaron's, . "]
['applied optoelectronics, . - ']
['aaon, . - ']
['advance auto parts  advance auto parts  w/i']
['apple . - ']
['american assets trust, . ']
['almaden minerals, ltd. common shares']
['atlas air worldwide holdings - ']
['axon enterprise, . - ']
['alliancebernstein holding l.p.  ']
['abb ltd ']
['abbvie . ']
['amerisourcebergen  ']
['ameris bancorp - ']
['abeona therapeutics . - ']
['ambev s.a. american  (each representing 1 common share)']
['asbury automotive   ']
['arca biopharma, . - ']
['abm industries orporated ']
['abiomed, . - ']
['arbor realty trust ']
['abbott laboratories ']
['allegiance bancshares, . - ']
['arbutus biopharma  - ']
['associated capital , . ']
['arcosa, .  ']
['acadia pharmaceuticals . - ']
['acamar partners acquisition corp. - class a ']
['acamar 

In [65]:
df_reuters['Description'][1]

'Walt Disney  has become the latest company to slash its advertising spending on Facebook Inc  as the social media giant faces an ad boycott over its handling of hate speech and controversial content, the Wall Street Journal reported on Saturday, citing people familiar with the situation.'